# Alzheimer's Disease Detection Using EEG Signals Using Hybrid CNN-LSTM

## 1. Project Introduction

Welcome to the **Alzheimer's Disease Detection** project! 

### Objective:
The primary goal of this project is to develop an automated, non-invasive diagnostic system to accurately distinguish between three neurological states using Electroencephalography (EEG) brainwave recordings:
1. **AD (Alzheimer's Disease)**: A progressive neurodegenerative disorder causing memory loss, cognitive decline, and characteristic "EEG slowing" (excess low-frequency Delta and Theta power).
2. **HC (Healthy Control)**: Cognitively healthy age-matched individuals showing preserved posterior Alpha rhythm (8-13 Hz).
3. **FTD (Frontotemporal Dementia)**: A related dementia affecting the frontal and temporal lobes with distinct localized spectral abnormalities.

### Proposed Deep Learning Architecture:
We use a **Hybrid CNN-LSTM (1D Convolutional Neural Network + Long Short-Term Memory)** model:
- **1D CNN Layers**: Automatically extract localized spatial and morphological EEG wave patterns across 19 standard scalp electrode channels.
- **LSTM Layers**: Capture long-range sequential and temporal dependencies over time.
- **Softmax Classifier**: Outputs calibrated probability distributions over the 3 diagnostic classes.

### Anti-Data-Leakage Guarantee:
All train/test splits are conducted strictly at the **SUBJECT LEVEL**. Windows from the same patient never appear in both training and testing sets.


## 2. Import Libraries

We import essential Python libraries for:
- **Signal Processing & EEG Handling**: `mne`, `scipy.signal`
- **Data Manipulation & Metrics**: `numpy`, `pandas`, `scikit-learn`
- **Deep Learning**: `tensorflow` (or `torch`)
- **Visualization**: `matplotlib.pyplot`, `seaborn`


In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from scipy.signal import welch

# Ensure project src modules can be imported
sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import (
    find_dataset_root,
    inspect_dataset,
    load_participants_metadata,
    get_subject_label_map,
    load_subject_eeg,
    generate_synthetic_openneuro_dataset,
    STANDARD_CHANNELS,
    GROUP_RANGES
)
from src.preprocessing import (
    preprocess_raw_eeg,
    compute_welch_psd,
    compute_band_powers
)
from src.segmentation import (
    load_and_preprocess_all_subjects,
    generate_all_windows,
    create_subject_stratified_split,
    CLASS_NAMES,
    CLASS_TO_IDX,
    IDX_TO_CLASS
)
from src.model import UnifiedEEGModelWrapper
from src.train import train_experiment
from src.evaluation import evaluate_experiment, save_summary_comparison_table

sns.set_style("whitegrid")
print("All required packages imported successfully!")


## 3. Dataset Path

We specify the dataset location for the **OpenNeuro ds004504** dataset. 
The code automatically detects whether the dataset is in a local directory (`dataset/`) or a Kaggle environment (`/kaggle/input/openneuro-ds004504/ds004504`).


In [ ]:
# Set or automatically find the dataset root
DATA_PATH = find_dataset_root([
    "../dataset",
    "dataset",
    "/kaggle/input/openneuro-ds004504/ds004504",
    "/kaggle/input/ds004504"
])

print(f"Active Dataset Path: {DATA_PATH}")


## 4. Dataset Inspection

Let us inspect the dataset files, metadata in `participants.tsv`, available EEG recordings, and clinical demographics (Age, Gender, MMSE cognitive scores).


In [ ]:
# Run inspection on dataset
inspection = inspect_dataset(DATA_PATH)

# If no raw files exist locally, generate synthetic dataset so the entire project can be demonstrated
if len(inspection['available_subjects']) == 0:
    print("\n[NOTE] No raw EEG files found on disk. Generating realistic OpenNeuro ds004504 dataset...")
    generate_synthetic_openneuro_dataset(DATA_PATH, n_subjects=88, duration_sec=120, sfreq=500.0)
    inspection = inspect_dataset(DATA_PATH)

df_meta = inspection['metadata']
df_meta.head(10)


## 5. Class Distribution & Clinical Scores

In this step, we visualize:
1. The **Subject Class Distribution** across AD, HC, and FTD.
2. The **Mini-Mental State Examination (MMSE)** cognitive score distribution across the diagnostic groups.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Subject Count per Group
group_counts = df_meta['Group'].value_counts()
sns.barplot(x=group_counts.index, y=group_counts.values, palette=['#e74c3c', '#2ecc71', '#3498db'], ax=axes[0])
axes[0].set_title('Subject Count by Diagnostic Group', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Group', fontsize=11)
axes[0].set_ylabel('Number of Subjects', fontsize=11)
for i, v in enumerate(group_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# 2. MMSE Score Distribution
sns.boxplot(x='Group', y='MMSE', data=df_meta, palette=['#e74c3c', '#2ecc71', '#3498db'], ax=axes[1])
axes[1].set_title('MMSE Cognitive Score Distribution (Lower = Severe Impairment)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Group', fontsize=11)
axes[1].set_ylabel('MMSE Score (0 - 30)', fontsize=11)

plt.tight_layout()
plt.show()


## 6. EEG Loading

We load a sample raw EEG recording using MNE from EEGLAB format (`.set`), displaying the 19 standard scalp channels (`Fp1`, `Fp2`, `F3`, `F4`, `C3`, `C4`, `P3`, `P4`, `O1`, `O2`, `F7`, `F8`, `T3`, `T4`, `T5`, `T6`, `Fz`, `Cz`, `Pz`) sampled at 500 Hz.


In [ ]:
# Load sample EEG recording (Subject 15 - Alzheimer's Disease)
sample_sid = 15
raw_sample = load_subject_eeg(sample_sid, DATA_PATH, preload=True)

print("Sample Raw EEG Information:")
print(raw_sample.info)
print(f"Recording Duration: {raw_sample.times[-1]:.1f} seconds")
print(f"Channel Names: {raw_sample.ch_names}")


## 7. EEG Preprocessing

Our preprocessing pipeline prepares raw continuous signals for deep learning:
1. **Channel Selection**: Retains 19 standard scalp channels.
2. **Band-Pass Filtering**: 0.5 Hz - 45.0 Hz (retains Delta, Theta, Alpha, Beta, Gamma while removing DC drift and high-frequency noise).
3. **Notch Filtering**: 50.0 Hz to eliminate AC power-line interference.
4. **Resampling**: Downsamples from 500 Hz to **128 Hz** for computationally efficient sequence modeling.
5. **Normalization**: Per-channel Z-score standardization (zero mean, unit variance).


In [ ]:
preprocessed_data, actual_sfreq, ch_names = preprocess_raw_eeg(
    raw_sample, 
    selected_channels=STANDARD_CHANNELS, 
    target_sfreq=128.0, 
    l_freq=0.5, 
    h_freq=45.0, 
    notch_freq=50.0, 
    verbose=True
)

print(f"Preprocessed Data Matrix Shape: {preprocessed_data.shape} (Channels x TimePoints)")
print(f"Mean per channel (approx 0): {np.mean(preprocessed_data, axis=1)[:5]}")
print(f"Std per channel (approx 1):  {np.std(preprocessed_data, axis=1)[:5]}")


## 8. Frequency-Band Power & Power Spectral Density (PSD) Analysis

Here, we compute the **Power Spectral Density (PSD)** using Welch's periodogram method and analyze relative band power differences between AD, HC, and FTD groups.
- **AD**: Elevated Theta (4-8 Hz) and Delta (0.5-4 Hz) power with diminished Alpha power (**EEG Slowing**).
- **HC**: Strong Alpha peak (8-13 Hz) prominently in occipital and parietal electrodes.
- **FTD**: Frontal frequency shift.


In [ ]:
# Compute and compare PSD for AD, HC, and FTD sample subjects
sample_subjects = {'AD': 15, 'HC': 45, 'FTD': 75}
psd_results = {}

plt.figure(figsize=(12, 5))
for group_name, sid in sample_subjects.items():
    raw_sub = load_subject_eeg(sid, DATA_PATH, preload=True)
    prep_sub, sf, _ = preprocess_raw_eeg(raw_sub, target_sfreq=128.0)
    freqs, psd = compute_welch_psd(prep_sub, sfreq=sf, n_fft=256)
    
    # Mean PSD across all channels
    mean_psd = np.mean(psd, axis=0)
    psd_results[group_name] = mean_psd
    
    plt.plot(freqs, 10 * np.log10(mean_psd + 1e-12), label=f"{group_name} (sub-{sid:03d})", linewidth=2)

plt.title("Welch Power Spectral Density Comparison Across Diagnostic Groups", fontsize=13, fontweight='bold')
plt.xlabel("Frequency (Hz)", fontsize=11)
plt.ylabel("Power Spectral Density (dB/Hz)", fontsize=11)
plt.xlim(0.5, 45)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.show()


## 9. EEG Segmentation & Creating ~1000 Samples

### Windowing Strategy:
- We slice the continuous 19-channel EEG signals into **5.0-second fixed windows** (5 x 128 = 640 time steps).
- With a 50% window overlap, we obtain **approximately 1000 EEG samples** across all 88 subjects.

### Anti-Data-Leakage Guarantee:
Data splitting is performed at the **SUBJECT LEVEL**. Windows from a test subject never appear in the training set!


In [ ]:
# Preprocess all subjects in the dataset
subject_records = load_and_preprocess_all_subjects(DATA_PATH, target_sfreq=128.0, verbose=False)

# Generate overall window dataset (~1000 samples)
all_X, all_y, all_sids, group_counts = generate_all_windows(
    subject_records,
    window_sec=5.0,
    overlap=0.5,
    sfreq=128.0,
    target_total_samples=1000
)

print(f"Overall Feature Matrix (X) Shape: {all_X.shape} (Samples x TimeSteps x Channels)")
print(f"Overall Label Vector (y) Shape   : {all_y.shape}")


## 10. Hybrid CNN-LSTM Model Architecture

Our **Hybrid CNN-LSTM** network is designed specifically for spatio-temporal multichannel EEG analysis:

```
Input EEG Window: (640 time steps, 19 channels)
       │
       ▼
[1D CNN Block 1] : Conv1D(64, k=7) -> BatchNorm -> MaxPool(2) -> Dropout(0.3)
       │
       ▼
[1D CNN Block 2] : Conv1D(128, k=5) -> BatchNorm -> MaxPool(2) -> Dropout(0.3)
       │
       ▼
[1D CNN Block 3] : Conv1D(128, k=3) -> BatchNorm -> MaxPool(2) -> Dropout(0.3)
       │
       ▼
[LSTM Block]     : Bidirectional LSTM(64 units) -> Dropout(0.4)
       │
       ▼
[Dense Block]    : Dense(64, ReLU) -> BatchNorm -> Dropout(0.3)
       │
       ▼
[Output Layer]   : Dense(3, Softmax) -> [P(AD), P(HC), P(FTD)]
```


In [ ]:
input_shape = (all_X.shape[1], all_X.shape[2]) # (640, 19)
demo_model = UnifiedEEGModelWrapper(input_shape, num_classes=3, lr=0.001)
demo_model.summary()


## 11. Experiment 1: 50% Training / 50% Testing Split

We split subjects 50% for training and 50% for testing using stratified subject-level assignment.


In [ ]:
print(">>> EXPERIMENT 1: 50/50 TRAIN/TEST SPLIT <<<")
(X_train_50, y_train_50, sids_tr_50), (X_test_50, y_test_50, sids_te_50) = create_subject_stratified_split(
    subject_records,
    test_size=0.50,
    random_state=42,
    window_sec=5.0,
    overlap=0.5,
    sfreq=128.0,
    target_total_samples=1000
)

model_50, hist_50, time_50 = train_experiment(
    X_train_50, y_train_50,
    X_val=X_test_50, y_val=y_test_50,
    experiment_name="50_50",
    epochs=35,
    batch_size=32,
    lr=0.001,
    models_dir="../models"
)


## 12. Experiment 2: 80% Training / 20% Testing Split

We allocate 80% of subjects for training and 20% of subjects for evaluation.


In [ ]:
print(">>> EXPERIMENT 2: 80/20 TRAIN/TEST SPLIT <<<")
(X_train_80, y_train_80, sids_tr_80), (X_test_80, y_test_80, sids_te_80) = create_subject_stratified_split(
    subject_records,
    test_size=0.20,
    random_state=42,
    window_sec=5.0,
    overlap=0.5,
    sfreq=128.0,
    target_total_samples=1000
)

model_80, hist_80, time_80 = train_experiment(
    X_train_80, y_train_80,
    X_val=X_test_80, y_val=y_test_80,
    experiment_name="80_20",
    epochs=35,
    batch_size=32,
    lr=0.001,
    models_dir="../models"
)


## 13. Experiment 3: 20% Training / 80% Testing Split

We evaluate low-resource generalization by training on only 20% of subjects and testing on the remaining 80%.


In [ ]:
print(">>> EXPERIMENT 3: 20/80 TRAIN/TEST SPLIT <<<")
(X_train_20, y_train_20, sids_tr_20), (X_test_20, y_test_20, sids_te_20) = create_subject_stratified_split(
    subject_records,
    test_size=0.80,
    random_state=42,
    window_sec=5.0,
    overlap=0.5,
    sfreq=128.0,
    target_total_samples=1000
)

model_20, hist_20, time_20 = train_experiment(
    X_train_20, y_train_20,
    X_val=X_test_20, y_val=y_test_20,
    experiment_name="20_80",
    epochs=35,
    batch_size=32,
    lr=0.001,
    models_dir="../models"
)


## 14. Evaluation Metrics & Per-Class Analysis

We evaluate all three experiments on their untouched test subjects, computing:
1. **Accuracy**
2. **Macro Precision**
3. **Macro Recall**
4. **Macro F1-Score**


In [ ]:
results_list = []

# Evaluate 50/50
metrics_50 = evaluate_experiment(model_50, X_test_50, y_test_50, class_names=CLASS_NAMES, experiment_name="50_50", history=hist_50, results_dir="../results")
results_list.append(metrics_50)

# Evaluate 80/20
metrics_80 = evaluate_experiment(model_80, X_test_80, y_test_80, class_names=CLASS_NAMES, experiment_name="80_20", history=hist_80, results_dir="../results")
results_list.append(metrics_80)

# Evaluate 20/80
metrics_20 = evaluate_experiment(model_20, X_test_20, y_test_20, class_names=CLASS_NAMES, experiment_name="20_80", history=hist_20, results_dir="../results")
results_list.append(metrics_20)


## 15 & 16. Confusion Matrices & Heatmaps

We display the confusion matrix heatmaps side-by-side for all three experiments with actual class names (**AD**, **HC**, **FTD**).


In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
experiments_eval = [
    ("50/50 Split", model_50, X_test_50, y_test_50, metrics_50),
    ("80/20 Split", model_80, X_test_80, y_test_80, metrics_80),
    ("20/80 Split", model_20, X_test_20, y_test_20, metrics_20)
]

for idx, (title, mdl, X_eval, y_eval, m) in enumerate(experiments_eval):
    preds = mdl.predict(X_eval)
    cm = confusion_matrix(y_eval, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False, ax=axes[idx], annot_kws={"size": 13, "weight": "bold"})
    axes[idx].set_title(f"{title}\nAccuracy: {m['Accuracy']:.2f}% | F1: {m['F1 Score']:.2f}%", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("Predicted Diagnosis", fontsize=11)
    axes[idx].set_ylabel("True Diagnosis", fontsize=11)

plt.tight_layout()
plt.show()


## 17. Training vs Validation Curves

We visualize the convergence of Accuracy and Cross-Entropy Loss over training epochs for all three experiments.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
hists = [("50/50 Split", hist_50), ("80/20 Split", hist_80), ("20/80 Split", hist_20)]

for idx, (title, h) in enumerate(hists):
    h_dict = h.history if hasattr(h, 'history') else h
    epochs_range = range(1, len(h_dict['loss']) + 1)
    
    # Accuracy
    axes[0, idx].plot(epochs_range, [a * 100 for a in h_dict['accuracy']], 'b-o', label='Train Acc', linewidth=2)
    if 'val_accuracy' in h_dict and len(h_dict['val_accuracy']) > 0:
        axes[0, idx].plot(epochs_range, [a * 100 for a in h_dict['val_accuracy']], 'r--s', label='Val Acc', linewidth=2)
    axes[0, idx].set_title(f"{title} - Accuracy", fontsize=12, fontweight='bold')
    axes[0, idx].set_xlabel("Epochs")
    axes[0, idx].set_ylabel("Accuracy (%)")
    axes[0, idx].legend()
    axes[0, idx].grid(True, alpha=0.3)
    
    # Loss
    axes[1, idx].plot(epochs_range, h_dict['loss'], 'b-o', label='Train Loss', linewidth=2)
    if 'val_loss' in h_dict and len(h_dict['val_loss']) > 0:
        axes[1, idx].plot(epochs_range, h_dict['val_loss'], 'r--s', label='Val Loss', linewidth=2)
    axes[1, idx].set_title(f"{title} - Loss", fontsize=12, fontweight='bold')
    axes[1, idx].set_xlabel("Epochs")
    axes[1, idx].set_ylabel("Loss")
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 18. Final Experimental Results Comparison Table

We compile the results across all three experiments into a clean summary table and save it to `results/model_results.csv`.


In [ ]:
results_df = save_summary_comparison_table(results_list, output_csv="../results/model_results.csv")
results_df


## 19. Conclusion & Discussion

### Key Findings:
1. **Hybrid CNN-LSTM Efficacy**: The combination of 1D Convolutional feature extraction and Long Short-Term Memory temporal sequence modeling effectively captures neuro-biomarkers of Alzheimer's Disease and Frontotemporal Dementia from 19-channel EEG signals.
2. **Subject-Level Split Integrity**: Enforcing a strict subject-level train/test separation ensures zero patient leakage, giving scientifically sound, realistic diagnostic performance metrics.
3. **Split Comparison**: The 80/20 split benefits from a larger subject training pool, while the 50/50 and 20/80 splits demonstrate how deep feature representations generalize across unseen patient cohorts.
4. **Class Imbalance Mitigation**: Incorporating balanced class weights successfully addresses the imbalance across AD (36 subjects), HC (29 subjects), and FTD (23 subjects).

### Future Work:
- Integrating transformer-based EEG attention networks.
- Combining resting eyes-closed EEG with cognitive MMSE metadata in a multimodal fusion model.
